# Quishing Detection - Revised Notebook

Notebook này được chuyển từ file `quishing_detect_final_revised.py` sang dạng `.ipynb` để bạn dễ chạy, dễ báo cáo và dễ chú thích.

## Mục tiêu chính
- Bổ sung **GridSearchCV** và **RandomizedSearchCV** chạy thật.
- Giữ đúng **6 mô hình** theo bài báo: Logistic Regression, Decision Tree, Random Forest, GaussianNB, LightGBM, XGBoost.
- Phần demo tạo QR đúng cấu hình paper: **version 13, 69×69, error correction LOW, border 0**.
- Xuất bảng **Môi trường thực nghiệm**.
- Giữ **app demo** nhưng nhấn mạnh đây chỉ là **minh họa ứng dụng**, không phải đánh giá chính thức.

## Cách chạy
Chạy lần lượt từ trên xuống dưới. Nếu muốn giảm thời gian chạy, hãy chỉnh các biến `RUN_*` ở cell cấu hình.


## Cell 1 - Import thư viện

**Mục đích:** nạp toàn bộ thư viện dùng cho pipeline, search, đánh giá, QR demo, explainability và app demo.

**Đầu ra mong đợi:** in ra trạng thái các thư viện tùy chọn như `ipywidgets`, `psutil`, `shap`.

**Lưu ý tham số:**
- `RANDOM_STATE = 42`: cố định seed để tái lập kết quả.
- `plt.rcParams`: cấu hình mặc định cho biểu đồ.


In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# %cd /content/drive/MyDrive/Learning/Master/7-ATHA517-Học Máy Trong An Toàn Thông Tin/Code

In [3]:
# %ls

In [4]:
# !pip install numpy pandas matplotlib scikit-learn xgboost lightgbm

In [5]:
# # Nếu môi trường chưa có các thư viện này thì bỏ comment dòng dưới để cài đặt

# !pip install -q numpy pandas matplotlib scikit-learn xgboost lightgbm joblib qrcode[pil] pillow
# !pip install -q ipywidgets  # chỉ cần nếu muốn dùng widget tương tác trong notebook

In [6]:
# Cell 1 - Import thư viện
#!/usr/bin/env python
# coding: utf-8

# Quishing ML Pipeline (Revised)
#
# Bản này sửa theo các yêu cầu sau:
# 1) Bổ sung block GridSearchCV và RandomizedSearchCV thật sự.
# 2) Giữ đúng 6 mô hình theo bài báo: Logistic Regression, Decision Tree,
#    Random Forest, GaussianNB, LightGBM, XGBoost.
# 3) Phần demo tạo QR dùng đúng cấu hình paper: version=13, size=69x69,
#    error correction=LOW, border=0, box_size=1.
# 4) Thêm bảng “Môi trường thực nghiệm”: Python, sklearn, xgboost,
#    lightgbm, CPU/RAM, thời gian train.
# 5) Giữ app demo nhưng ghi rõ đây chỉ là phần minh họa ứng dụng,
#    không phải đánh giá chính thức.
#
# Gợi ý chạy:
#   python quishing_detect_final_revised.py
#
# Nếu muốn giảm thời gian chạy, có thể tắt bớt ở phần CONFIG:
#   RUN_CV = False
#   RUN_GRID_SEARCH = False
#   RUN_RANDOMIZED_SEARCH = False

from __future__ import annotations

from pathlib import Path
import os
import sys
import time
import json
import pickle
import random
import warnings
import platform
from typing import Any, Dict, Iterable, Optional, Tuple

# --- Windows/joblib safe parallel config ---
# Tránh lỗi joblib/loky trên Windows khi dò physical cores và tránh nested parallelism.
CPU_COUNT = os.cpu_count() or 2
MODEL_N_JOBS = max(1, min(4, CPU_COUNT // 2))
CV_N_JOBS = 1
SEARCH_N_JOBS = 1

os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(MODEL_N_JOBS))
os.environ.setdefault("OMP_NUM_THREADS", str(MODEL_N_JOBS))
os.environ.setdefault("MKL_NUM_THREADS", str(MODEL_N_JOBS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(MODEL_N_JOBS))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
    RandomizedSearchCV,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    make_scorer,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    ConfusionMatrixDisplay,
    average_precision_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

from lightgbm import LGBMClassifier
import lightgbm
from xgboost import XGBClassifier
import xgboost

try:
    import qrcode
    from PIL import Image
except ImportError as e:
    raise ImportError(
        "Thiếu thư viện cho phần tạo QR. Hãy cài qrcode[pil] và pillow trước khi chạy."
    ) from e

try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    psutil = None
    HAS_PSUTIL = False

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    widgets = None
    HAS_WIDGETS = False

try:
    import shap
    HAS_SHAP = True
except ImportError:
    shap = None
    HAS_SHAP = False

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("Libraries imported successfully.")
print("ipywidgets available:", HAS_WIDGETS)
print("psutil available    :", HAS_PSUTIL)
print("shap available      :", HAS_SHAP)
print("MODEL_N_JOBS        :", MODEL_N_JOBS)
print("CV_N_JOBS           :", CV_N_JOBS)
print("SEARCH_N_JOBS       :", SEARCH_N_JOBS)


Libraries imported successfully.
ipywidgets available: True
psutil available    : True
shap available      : False
MODEL_N_JOBS        : 4
CV_N_JOBS           : 1
SEARCH_N_JOBS       : 1


## Cell 2 - Cấu hình thực nghiệm

**Mục đích:** khai báo đường dẫn dữ liệu, tham số chia tập, số fold, cấu hình search, cờ bật/tắt từng stage và cấu hình QR đúng paper.

**Tham số quan trọng:**
- `TEST_SIZE = 0.20`: chia train/test theo tỉ lệ 80/20.
- `CV_FOLDS = [5, 10, 15, 20]`: các fold dùng để so sánh cross-validation.
- `SEARCH_CV_FOLD = 10`: số fold dùng riêng cho GridSearchCV / RandomizedSearchCV.
- `SEARCH_SCORING = "f1"`: metric tối ưu khi search.
- `RANDOM_SEARCH_N_ITER = 12`: số lần lấy mẫu cho RandomizedSearchCV.
- `RUN_*`: bật/tắt từng phần để tiết kiệm thời gian.
- `QR_VERSION = 13`, `QR_MODULE_SIZE = 69`, `QR_BORDER = 0`, `QR_BOX_SIZE = 1`: cấu hình QR đồng bộ với paper.
- `TOP_K_PIXELS = 500`: số pixel quan trọng nhất giữ lại ở thí nghiệm rút gọn đặc trưng.


In [7]:
# Cell 2 - Cấu hình thực nghiệm
# =========================
# 1) CONFIG
# =========================
QR_IMAGE_CANDIDATES = [
    "./dataset/qr_codes_29.pickle",
    "/content/drive/MyDrive/Learning/Master/7-ATHA517-Học Máy Trong An Toàn Thông Tin/Code/Dataset/qr_codes_29.pickle",
    "/content/drive/MyDrive/MINAS/THẠC SĨ ATTT KMA/HỌC MÁY/Quishing/Dataset/qr_codes_29.pickle",
    "/mnt/data/qr_codes_29.pickle",
]

QR_LABEL_CANDIDATES = [
    "./dataset/qr_codes_29_labels.pickle",
    "/content/drive/MyDrive/Learning/Master/7-ATHA517-Học Máy Trong An Toàn ThôNG Tin/Code/Dataset/qr_codes_29_labels.pickle",
    "/content/drive/MyDrive/MINAS/THẠC SĨ ATTT KMA/HỌC MÁY/Quishing/Dataset/qr_codes_29_labels.pickle",
    "/mnt/data/qr_codes_29_labels.pickle",
]

TEST_SIZE = 0.20
CV_FOLDS = [5, 10, 15, 20]
SEARCH_CV_FOLD = 10
SEARCH_SCORING = "f1"
RANDOM_SEARCH_N_ITER = 12

RUN_PAPER_CONFIG = True
RUN_CUSTOM_CONFIG = True
RUN_NO_CV = True
RUN_CV = True
RUN_GRID_SEARCH = True
RUN_RANDOMIZED_SEARCH = True
RUN_REDUCED_FEATURE_EXPERIMENTS = True

LABEL_NAME_MAP = {0: "benign", 1: "malicious"}
PREFERRED_DEMO_FOLD = 10
PREFERRED_DEMO_SEARCH_METHOD = "randomized"
EXPLAINABILITY_SOURCE_SEARCH_METHOD = "randomized"
EXPLAINABILITY_SOURCE_CONFIG_NAME = "paper_params"
EXPLAINABILITY_SOURCE_FOLD = 10

# QR đúng theo bài báo
QR_VERSION = 13
QR_MODULE_SIZE = 69
QR_ERROR_CORRECTION = qrcode.constants.ERROR_CORRECT_L
QR_BOX_SIZE = 1
QR_BORDER = 0
QR_TARGET_SHAPE = (QR_MODULE_SIZE, QR_MODULE_SIZE)

OUTPUT_DIR = Path("./outputs")
EDA_DIR = OUTPUT_DIR / "eda"
MODEL_DIR = OUTPUT_DIR / "models"
RESULT_DIR = OUTPUT_DIR / "results"
GENERATED_QR_DIR = OUTPUT_DIR / "generated_qr"
EXPLAIN_DIR = OUTPUT_DIR / "explainability"
PLOT_DIR = OUTPUT_DIR / "plots"
APP_DIR = OUTPUT_DIR / "app"

for d in [OUTPUT_DIR, EDA_DIR, MODEL_DIR, RESULT_DIR, GENERATED_QR_DIR, EXPLAIN_DIR, PLOT_DIR, APP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TOP_IMPORTANCE_N = 25
TOP_K_PIXELS = 500
SHAP_SAMPLE_SIZE = 200
TOP_DIAGNOSTIC_MODELS = 3

print(f"Output directory: {OUTPUT_DIR.resolve()}")


Output directory: C:\Users\httnghia\Downloads\ATTT_ML\outputs


## Cell 3 - Hàm tiện ích chung

**Mục đích:** gom các hàm dùng nhiều lần như:
- tìm file dữ liệu,
- đọc pickle,
- làm phẳng ảnh QR,
- tính metric,
- lưu model + metadata,
- gom bảng giải thích chỉ số,
- thu thập thông tin môi trường thực nghiệm.

**Điểm đáng chú ý:**
- `collect_experiment_environment(...)` sẽ tạo bảng môi trường để đưa thẳng vào báo cáo.
- `save_trained_model(...)` lưu cả model và metadata để app/demo dùng lại.


In [8]:
# Cell 3 - Helper functions
# =========================
# 2) HELPER FUNCTIONS
# =========================
def resolve_existing_path(candidates: Iterable[str]) -> Path:
    for p in candidates:
        path = Path(p)
        if path.exists():
            return path
    raise FileNotFoundError(
        "Không tìm thấy file trong danh sách candidates. Hãy cập nhật đường dẫn ở cell CONFIG."
    )


def load_pickle(path: Path) -> Any:
    with open(path, "rb") as f:
        return pickle.load(f)


def ensure_numpy(arr: Any) -> np.ndarray:
    return np.asarray(arr)


def flatten_qr_images(qr_images: np.ndarray) -> Tuple[np.ndarray, Tuple[int, int]]:
    qr_images = ensure_numpy(qr_images)
    if qr_images.ndim == 3:
        n, h, w = qr_images.shape
        return qr_images.reshape(n, h * w), (h, w)
    if qr_images.ndim == 2:
        n, d = qr_images.shape
        side = int(np.sqrt(d))
        image_shape = (side, side) if side * side == d else (1, d)
        return qr_images, image_shape
    raise ValueError(f"QR images phải có shape (N,H,W) hoặc (N,D), nhận được: {qr_images.shape}")


def get_positive_scores(model: Any, X: np.ndarray) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
        return scores
    return model.predict(X)


def evaluate_model(model: Any, X: np.ndarray, y_true: np.ndarray) -> Dict[str, float]:
    y_pred = model.predict(X)
    y_score = get_positive_scores(model, X)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_score),
    }


def make_scoring_dict() -> Dict[str, Any]:
    return {
        "accuracy": "accuracy",
        "precision": make_scorer(precision_score, zero_division=0),
        "recall": make_scorer(recall_score, zero_division=0),
        "f1_score": make_scorer(f1_score, zero_division=0),
        "auc": "roc_auc",
    }


def sanitize_name(name: str) -> str:
    return "".join(ch if ch.isalnum() or ch in ["_", "-"] else "_" for ch in name)


def to_builtin(value: Any) -> Any:
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.ndarray, list, tuple)):
        return [to_builtin(v) for v in list(value)]
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    return value


def save_trained_model(model: Any, save_path: Path, metadata: Optional[dict] = None) -> None:
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    payload = {"model": model, "metadata": metadata or {}}
    joblib.dump(payload, save_path)


def summarize_cv_scores(cv_result: dict) -> Dict[str, float]:
    summary = {}
    for key, values in cv_result.items():
        if key.startswith("test_"):
            metric_name = key.replace("test_", "")
            summary[f"cv_{metric_name}_mean"] = float(np.mean(values))
            summary[f"cv_{metric_name}_std"] = float(np.std(values))
    return summary


def decode_label_name(label_value: Any, label_name_map: Optional[dict] = None) -> str:
    label_name_map = label_name_map or LABEL_NAME_MAP
    try:
        label_value = int(label_value)
    except Exception:
        return str(label_value)
    return label_name_map.get(label_value, f"class_{label_value}")


def load_model_bundle(saved_model_path: str | Path) -> dict:
    bundle = joblib.load(saved_model_path)
    if isinstance(bundle, dict) and "model" in bundle:
        return bundle
    return {"model": bundle, "metadata": {}}


def estimate_search_space_size(param_grid: Dict[str, list]) -> int:
    total = 1
    for values in param_grid.values():
        total *= max(len(values), 1)
    return total


def fit_searcher(
    base_model: Any,
    param_space: Dict[str, list],
    method: str,
    cv: Any,
    scoring: str = SEARCH_SCORING,
    n_iter: int = RANDOM_SEARCH_N_ITER,
) -> Any:
    if method == "grid":
        return GridSearchCV(
            estimator=clone(base_model),
            param_grid=param_space,
            scoring=scoring,
            cv=cv,
            n_jobs=SEARCH_N_JOBS,
            refit=True,
            return_train_score=False,
        )
    if method == "randomized":
        total_space = estimate_search_space_size(param_space)
        return RandomizedSearchCV(
            estimator=clone(base_model),
            param_distributions=param_space,
            n_iter=min(n_iter, total_space),
            scoring=scoring,
            cv=cv,
            n_jobs=SEARCH_N_JOBS,
            refit=True,
            random_state=RANDOM_STATE,
            return_train_score=False,
        )
    raise ValueError(f"search method không hợp lệ: {method}")


def build_metric_glossary_df() -> pd.DataFrame:
    rows = [
        {
            "column_name": "accuracy / test_accuracy / cv_accuracy_mean",
            "meaning": "Tỷ lệ mẫu được dự đoán đúng trên toàn bộ tập đánh giá.",
            "how_to_read": "Càng cao càng tốt; nhưng nếu dữ liệu lệch lớp thì không nên dùng accuracy một mình.",
        },
        {
            "column_name": "precision / test_precision / cv_precision_mean",
            "meaning": "Trong các mẫu bị dự đoán là positive (mặc định class 1), có bao nhiêu mẫu đúng thật.",
            "how_to_read": "Quan trọng khi muốn giảm false positive.",
        },
        {
            "column_name": "recall / test_recall / cv_recall_mean",
            "meaning": "Trong các mẫu positive thật, model bắt được bao nhiêu mẫu.",
            "how_to_read": "Quan trọng khi muốn giảm false negative.",
        },
        {
            "column_name": "f1_score / test_f1_score / cv_f1_score_mean",
            "meaning": "Trung bình điều hòa giữa precision và recall.",
            "how_to_read": "Hữu ích khi cần cân bằng giữa precision và recall.",
        },
        {
            "column_name": "auc / test_auc / cv_auc_mean",
            "meaning": "ROC-AUC: khả năng tách 2 lớp trên nhiều ngưỡng quyết định.",
            "how_to_read": "Gần 1.0 là tốt; 0.5 là gần như ngẫu nhiên.",
        },
        {
            "column_name": "cv_*_std",
            "meaning": "Độ lệch chuẩn của metric qua các folds.",
            "how_to_read": "Càng nhỏ càng ổn định giữa các folds.",
        },
        {
            "column_name": "search_method",
            "meaning": "Phương pháp tối ưu siêu tham số thực sự đã dùng.",
            "how_to_read": "Giá trị có thể là grid hoặc randomized.",
        },
        {
            "column_name": "search_best_score",
            "meaning": "Điểm CV tốt nhất theo metric scoring dùng cho search.",
            "how_to_read": "Trong file này mặc định tối ưu theo F1-score.",
        },
        {
            "column_name": "search_best_params",
            "meaning": "Bộ siêu tham số tốt nhất tìm được bởi GridSearchCV hoặc RandomizedSearchCV.",
            "how_to_read": "Dùng để báo cáo và tái lập kết quả.",
        },
        {
            "column_name": "elapsed_seconds",
            "meaning": "Thời gian train + evaluate hoặc search + refit.",
            "how_to_read": "Dùng để so sánh chi phí tính toán giữa các model.",
        },
        {
            "column_name": "top_k_pixels",
            "meaning": "Số pixel quan trọng nhất được giữ lại sau feature selection.",
            "how_to_read": "Ví dụ 500 nghĩa là chỉ giữ 500 pixel quan trọng nhất.",
        },
        {
            "column_name": "reduced_feature_ratio",
            "meaning": "Tỷ lệ feature sau rút gọn so với toàn bộ pixel ban đầu.",
            "how_to_read": "Ví dụ 0.10 nghĩa là chỉ giữ lại 10% feature.",
        },
        {
            "column_name": "prob_class_1",
            "meaning": "Xác suất/score dự đoán cho class 1.",
            "how_to_read": "Nếu LABEL_NAME_MAP mặc định thì class 1 = malicious.",
        },
    ]
    return pd.DataFrame(rows)


def collect_experiment_environment(
    image_shape: Tuple[int, int],
    X_train: np.ndarray,
    X_test: np.ndarray,
    stage_frames: Optional[Dict[str, pd.DataFrame]] = None,
) -> pd.DataFrame:
    python_version = sys.version.split()[0]
    cpu_name = platform.processor() or platform.machine() or "unknown"
    logical_cores = os.cpu_count()
    ram_gb = None
    if HAS_PSUTIL:
        ram_gb = round(psutil.virtual_memory().total / (1024 ** 3), 2)

    rows = [
        {"item": "Python", "value": python_version},
        {"item": "scikit-learn", "value": sklearn.__version__},
        {"item": "xgboost", "value": xgboost.__version__},
        {"item": "lightgbm", "value": lightgbm.__version__},
        {"item": "numpy", "value": np.__version__},
        {"item": "pandas", "value": pd.__version__},
        {"item": "CPU", "value": cpu_name},
        {"item": "CPU logical cores", "value": logical_cores},
        {"item": "RAM (GB)", "value": ram_gb if ram_gb is not None else "psutil not installed"},
        {"item": "QR version", "value": QR_VERSION},
        {"item": "QR image size", "value": f"{image_shape[0]}x{image_shape[1]}"},
        {"item": "QR error correction", "value": "LOW"},
        {"item": "QR border", "value": QR_BORDER},
        {"item": "Train shape", "value": str(tuple(X_train.shape))},
        {"item": "Test shape", "value": str(tuple(X_test.shape))},
    ]

    stage_frames = stage_frames or {}
    for stage_name, frame in stage_frames.items():
        if frame is None or len(frame) == 0 or "elapsed_seconds" not in frame.columns:
            continue
        rows.append({
            "item": f"Training time - {stage_name} (seconds)",
            "value": round(float(frame["elapsed_seconds"].sum()), 4),
        })

    return pd.DataFrame(rows)


## Cell 4 - Khai báo bộ mô hình và search space

**Mục đích:** định nghĩa đúng **6 mô hình** theo bài báo và search space thật sự cho GridSearchCV / RandomizedSearchCV.

**6 mô hình giữ lại:**
1. LogisticRegression
2. DecisionTree
3. RandomForest
4. GaussianNB
5. LightGBM
6. XGBoost

**Đầu ra:**
- `model_sets`: gồm `paper_params` và `custom_params`
- `search_spaces`: không gian tham số dùng cho search thật


In [9]:
# Cell 4 - Model definitions và search spaces
# =========================
# 3) MODEL DEFINITIONS
# =========================
def build_model_sets(random_state: int = 42) -> Tuple[Dict[str, Dict[str, Any]], Dict[str, Dict[str, list]]]:
    # Đúng 6 mô hình theo paper
    paper_params = {
        "LogisticRegression": LogisticRegression(C=0.1, solver="liblinear", max_iter=1000, random_state=random_state),
        "DecisionTree": DecisionTreeClassifier(max_depth=3, min_samples_leaf=1, random_state=random_state),
        "RandomForest": RandomForestClassifier(max_depth=20, n_estimators=100, random_state=random_state, n_jobs=MODEL_N_JOBS),
        "GaussianNB": GaussianNB(),
        "LightGBM": LGBMClassifier(learning_rate=0.1, n_estimators=200, random_state=random_state, verbose=-1, n_jobs=MODEL_N_JOBS),
        "XGBoost": XGBClassifier(learning_rate=0.2, n_estimators=150, random_state=random_state, eval_metric="logloss", n_jobs=MODEL_N_JOBS),
    }

    custom_params = {
        "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=random_state),
        "DecisionTree": DecisionTreeClassifier(random_state=random_state, class_weight="balanced"),
        "RandomForest": RandomForestClassifier(n_estimators=200, random_state=random_state, class_weight="balanced", n_jobs=MODEL_N_JOBS),
        "GaussianNB": GaussianNB(),
        "LightGBM": LGBMClassifier(n_estimators=200, random_state=random_state, verbose=-1, n_jobs=MODEL_N_JOBS),
        "XGBoost": XGBClassifier(n_estimators=200, eval_metric="logloss", random_state=random_state, n_jobs=MODEL_N_JOBS),
    }

    # Search space thật sự cho cả GridSearchCV và RandomizedSearchCV
    search_spaces = {
        "LogisticRegression": {
            "C": [0.01, 0.1, 1, 10],
            "penalty": ["l2"],
            "solver": ["liblinear", "lbfgs"],
        },
        "DecisionTree": {
            "max_depth": [3, 5, 10, 20, None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 5],
        },
        "RandomForest": {
            "n_estimators": [100, 200, 400],
            "max_depth": [10, 20, None],
            "min_samples_leaf": [1, 2, 4],
            "min_samples_split": [2, 5, 10],
        },
        "GaussianNB": {
            "var_smoothing": [1e-12, 1e-11, 1e-10, 1e-9, 1e-8],
        },
        "LightGBM": {
            "learning_rate": [0.01, 0.05, 0.1],
            "n_estimators": [100, 200, 400],
            "num_leaves": [15, 31, 63],
            "max_depth": [-1, 10, 20],
        },
        "XGBoost": {
            "learning_rate": [0.01, 0.05, 0.1, 0.2],
            "n_estimators": [100, 150, 200, 400],
            "max_depth": [3, 6, 10],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0],
        },
    }

    model_sets = {}
    if RUN_PAPER_CONFIG:
        model_sets["paper_params"] = paper_params
    if RUN_CUSTOM_CONFIG:
        model_sets["custom_params"] = custom_params

    return model_sets, search_spaces


## Cell 5 - Các hàm train/evaluate/search

**Mục đích:** triển khai các stage huấn luyện chính:
- train không CV,
- train có CV nhiều fold,
- train với GridSearchCV hoặc RandomizedSearchCV.

**Đầu ra của mỗi stage:**
- bảng kết quả metric,
- thời gian chạy,
- model đã fit,
- file `.joblib` chứa model + metadata.


In [10]:
# Cell 5 - Training functions
# =========================
# 4) TRAINING FUNCTIONS
# =========================
def train_without_cv(
    model_sets: Dict[str, Dict[str, Any]],
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray,
    model_dir: Path,
) -> pd.DataFrame:
    results = []
    for config_name, models in model_sets.items():
        print(f"\n===== TRAIN NO-CV | CONFIG: {config_name} =====")
        for model_name, base_model in models.items():
            start = time.time()
            model = clone(base_model)
            model.fit(X_train, y_train)
            metrics = evaluate_model(model, X_test, y_test)
            elapsed = time.time() - start

            save_path = model_dir / "no_cv" / config_name / f"{sanitize_name(model_name)}.joblib"
            save_trained_model(
                model,
                save_path,
                metadata={
                    "stage": "no_cv",
                    "config_name": config_name,
                    "model_name": model_name,
                    "metrics": to_builtin(metrics),
                    "elapsed_seconds": float(elapsed),
                    "reference_image_stats": {
                        "min": float(np.min(X_train)),
                        "max": float(np.max(X_train)),
                        "mean": float(np.mean(X_train)),
                    },
                },
            )

            row = {
                "stage": "no_cv",
                "fold": 0,
                "search_method": "none",
                "config_name": config_name,
                "model_name": model_name,
                **metrics,
                "elapsed_seconds": elapsed,
                "saved_model_path": str(save_path),
            }
            results.append(row)
            print(f"{model_name:18s} | AUC={metrics['auc']:.4f} | F1={metrics['f1_score']:.4f} | saved={save_path.name}")
    return pd.DataFrame(results)


def train_with_cv(
    model_sets: Dict[str, Dict[str, Any]],
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray,
    cv_folds: Iterable[int],
    model_dir: Path,
) -> pd.DataFrame:
    scoring = make_scoring_dict()
    results = []

    for fold in cv_folds:
        print(f"\n================ CV = {fold} ================")
        skf = StratifiedKFold(n_splits=fold, shuffle=True, random_state=RANDOM_STATE)

        for config_name, models in model_sets.items():
            print(f"\n----- CONFIG: {config_name} -----")
            for model_name, base_model in models.items():
                start = time.time()
                model_for_cv = clone(base_model)
                cv_scores = cross_validate(
                    model_for_cv,
                    X_train,
                    y_train,
                    cv=skf,
                    scoring=scoring,
                    n_jobs=CV_N_JOBS,
                    return_train_score=False,
                )
                cv_summary = summarize_cv_scores(cv_scores)

                final_model = clone(base_model)
                final_model.fit(X_train, y_train)
                test_metrics = evaluate_model(final_model, X_test, y_test)
                elapsed = time.time() - start

                save_path = model_dir / f"cv_{fold}" / config_name / f"{sanitize_name(model_name)}.joblib"
                save_trained_model(
                    final_model,
                    save_path,
                    metadata={
                        "stage": "cv",
                        "fold": int(fold),
                        "search_method": "none",
                        "config_name": config_name,
                        "model_name": model_name,
                        "cv_summary": to_builtin(cv_summary),
                        "test_metrics": to_builtin(test_metrics),
                        "elapsed_seconds": float(elapsed),
                        "reference_image_stats": {
                            "min": float(np.min(X_train)),
                            "max": float(np.max(X_train)),
                            "mean": float(np.mean(X_train)),
                        },
                    },
                )

                row = {
                    "stage": "cv",
                    "fold": int(fold),
                    "search_method": "none",
                    "config_name": config_name,
                    "model_name": model_name,
                    **cv_summary,
                    **{f"test_{k}": v for k, v in test_metrics.items()},
                    "elapsed_seconds": elapsed,
                    "saved_model_path": str(save_path),
                }
                results.append(row)
                print(
                    f"{model_name:18s} | CV AUC={cv_summary['cv_auc_mean']:.4f} ± {cv_summary['cv_auc_std']:.4f} "
                    f"| Test AUC={test_metrics['auc']:.4f}"
                )

    return pd.DataFrame(results)


def train_with_search(
    model_sets: Dict[str, Dict[str, Any]],
    search_spaces: Dict[str, Dict[str, list]],
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray,
    search_cv_fold: int,
    method: str,
    model_dir: Path,
    scoring: str = SEARCH_SCORING,
) -> pd.DataFrame:
    results = []
    skf = StratifiedKFold(n_splits=search_cv_fold, shuffle=True, random_state=RANDOM_STATE)

    print(f"\n================ {method.upper()} SEARCH | CV = {search_cv_fold} ================")
    for config_name, models in model_sets.items():
        print(f"\n----- CONFIG: {config_name} -----")
        for model_name, base_model in models.items():
            if model_name not in search_spaces:
                print(f"{model_name:18s} | skipped (no search space)")
                continue

            start = time.time()
            searcher = fit_searcher(
                base_model=base_model,
                param_space=search_spaces[model_name],
                method=method,
                cv=skf,
                scoring=scoring,
                n_iter=RANDOM_SEARCH_N_ITER,
            )
            searcher.fit(X_train, y_train)

            best_model = searcher.best_estimator_
            test_metrics = evaluate_model(best_model, X_test, y_test)
            elapsed = time.time() - start
            best_params = to_builtin(searcher.best_params_)
            best_score = float(searcher.best_score_)

            save_path = model_dir / f"{method}_search_cv_{search_cv_fold}" / config_name / f"{sanitize_name(model_name)}.joblib"
            save_trained_model(
                best_model,
                save_path,
                metadata={
                    "stage": "search",
                    "fold": int(search_cv_fold),
                    "search_method": method,
                    "config_name": config_name,
                    "model_name": model_name,
                    "search_best_score": best_score,
                    "search_best_params": best_params,
                    "test_metrics": to_builtin(test_metrics),
                    "elapsed_seconds": float(elapsed),
                    "reference_image_stats": {
                        "min": float(np.min(X_train)),
                        "max": float(np.max(X_train)),
                        "mean": float(np.mean(X_train)),
                    },
                },
            )

            row = {
                "stage": "search",
                "fold": int(search_cv_fold),
                "search_method": method,
                "config_name": config_name,
                "model_name": model_name,
                "search_scoring": scoring,
                "search_best_score": best_score,
                "search_best_params": json.dumps(best_params, ensure_ascii=False),
                **{f"test_{k}": v for k, v in test_metrics.items()},
                "elapsed_seconds": elapsed,
                "saved_model_path": str(save_path),
            }
            results.append(row)
            print(
                f"{model_name:18s} | best_{scoring}={best_score:.4f} | Test AUC={test_metrics['auc']:.4f} | params={best_params}"
            )

    return pd.DataFrame(results)


## Cell 6 - Hàm hỗ trợ QR demo an toàn

**Mục đích:** tạo QR synthetic để minh họa ứng dụng mà không dùng payload nguy hiểm thật.

**Lưu ý quan trọng:**
- Đây chỉ là **application demo**.
- Không dùng để thay thế phần **đánh giá chính thức** trên test set / cross-validation.

**Tham số quan trọng:**
- `render_qr_to_array(...)` bắt buộc QR phải đúng `version=13`, `LOW`, `border=0`, `69x69`.
- `align_generated_array_to_reference(...)` căn chỉnh miền giá trị ảnh demo về gần dữ liệu train.


In [11]:
# Cell 6 - Synthetic QR demo helpers
# =========================
# 5) SAFE SYNTHETIC QR DEMO HELPERS
# =========================
SAFE_BENIGN_QR_PAYLOADS = [
    "https://example.com/news/academic-2026",
    "WIFI:T:WPA;S:CampusGuest;P:Safe12345;;",
    "BEGIN:VCARD\nFN:Lab Reception\nTEL:+84000000000\nEND:VCARD",
    "https://example.org/library/week-12",
]

SAFE_SIMULATED_MALICIOUS_QR_PAYLOADS = [
    "http://198.51.100.24/verify?id=8842",
    "https://login.example.invalid/update?t=reset",
    "http://203.0.113.7/billing?r=confirm",
    "https://account.example.invalid/open?n=signin",
]


def get_safe_qr_payload(qr_type: str = "benign", sample_index: int = 0) -> Tuple[str, str, int]:
    qr_type = str(qr_type).strip().lower()
    if qr_type in {"benign", "normal", "safe"}:
        canonical_type = "benign"
        pool = SAFE_BENIGN_QR_PAYLOADS
    elif qr_type in {"malicious", "suspicious", "simulated_malicious"}:
        canonical_type = "malicious"
        pool = SAFE_SIMULATED_MALICIOUS_QR_PAYLOADS
    else:
        raise ValueError("qr_type phải là 'benign' hoặc 'malicious'.")

    sample_index = int(sample_index) % len(pool)
    return canonical_type, pool[sample_index], sample_index


def render_qr_to_array(
    text: str,
    target_shape: Tuple[int, int] = QR_TARGET_SHAPE,
    version: int = QR_VERSION,
    error_correction: int = QR_ERROR_CORRECTION,
    box_size: int = QR_BOX_SIZE,
    border: int = QR_BORDER,
) -> Tuple[Image.Image, np.ndarray]:
    qr = qrcode.QRCode(
        version=version,
        error_correction=error_correction,
        box_size=box_size,
        border=border,
    )
    try:
        qr.add_data(text)
        qr.make(fit=False)
    except Exception as e:
        raise ValueError(
            "Payload không thể mã hóa bằng QR version 13 với error correction LOW và border 0. "
            "Hãy rút ngắn nội dung đầu vào cho phần demo."
        ) from e

    qr_img = qr.make_image(fill_color="black", back_color="white").convert("L")
    qr_arr = np.asarray(qr_img, dtype=np.uint8)

    if qr_arr.shape != target_shape:
        raise ValueError(
            f"QR render shape = {qr_arr.shape}, nhưng kỳ vọng đúng {target_shape}. "
            "Hãy kiểm tra lại version / border / box_size."
        )
    return qr_img, qr_arr


def align_generated_array_to_reference(qr_array: np.ndarray, reference_images: np.ndarray) -> np.ndarray:
    ref = ensure_numpy(reference_images).astype(np.float32)
    ref_min, ref_max = float(ref.min()), float(ref.max())

    if ref_max <= 1.0 + 1e-12:
        base = qr_array.astype(np.float32) / 255.0
    else:
        if ref_max - ref_min < 1e-12:
            base = np.full_like(qr_array, fill_value=ref_min, dtype=np.float32)
        else:
            base = qr_array.astype(np.float32) / 255.0 * (ref_max - ref_min) + ref_min

    inv = ref_min + ref_max - base
    ref_mean = float(ref.mean())
    chosen = base if abs(float(base.mean()) - ref_mean) <= abs(float(inv.mean()) - ref_mean) else inv
    return chosen.astype(reference_images.dtype)


def predict_single_qr_array(model: Any, qr_array: np.ndarray) -> Tuple[int, float]:
    X_input = qr_array.reshape(1, -1)
    pred_label = model.predict(X_input)[0]
    class1_score = float(get_positive_scores(model, X_input)[0])
    return int(pred_label), class1_score


def predict_generated_qr(
    model_bundle: dict,
    reference_images: np.ndarray,
    target_shape: Tuple[int, int],
    qr_type: str = "benign",
    sample_index: int = 0,
    label_name_map: Optional[dict] = None,
    save_dir: Optional[Path] = None,
    show_plot: bool = True,
) -> Dict[str, Any]:
    label_name_map = label_name_map or LABEL_NAME_MAP
    model = model_bundle["model"]

    qr_type, payload, sample_index = get_safe_qr_payload(qr_type=qr_type, sample_index=sample_index)
    qr_img, qr_arr_raw = render_qr_to_array(payload, target_shape=target_shape)
    qr_arr_for_model = align_generated_array_to_reference(qr_arr_raw, reference_images)

    pred_label, prob_class_1 = predict_single_qr_array(model, qr_arr_for_model)
    predicted_label_name = decode_label_name(pred_label, label_name_map=label_name_map)
    expected_label_name = "malicious" if qr_type == "malicious" else "benign"

    saved_image_path = None
    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        file_name = f"{qr_type}_sample_{sample_index:02d}.png"
        saved_image_path = save_dir / file_name
        qr_img.save(saved_image_path)

    if show_plot:
        plt.figure(figsize=(4, 4))
        plt.imshow(qr_arr_for_model, cmap="gray")
        plt.axis("off")
        plt.title(
            "Generated QR (application demo only)\n"
            f"expected={expected_label_name} | predicted={predicted_label_name}\n"
            f"P(class 1)={prob_class_1:.4f}"
        )
        plt.tight_layout()
        plt.show()

    return {
        "sample_type": qr_type,
        "sample_index": sample_index,
        "payload": payload,
        "expected_label_name": expected_label_name,
        "predicted_label_value": pred_label,
        "predicted_label_name": predicted_label_name,
        "prob_class_1": prob_class_1,
        "matches_expected_simulation": predicted_label_name.lower() == expected_label_name.lower(),
        "saved_qr_image_path": str(saved_image_path) if saved_image_path is not None else None,
    }


def select_best_model_row(
    no_cv_results_df: Optional[pd.DataFrame] = None,
    cv_results_df: Optional[pd.DataFrame] = None,
    search_results_df: Optional[pd.DataFrame] = None,
    preferred_fold: int = 10,
    preferred_search_method: str = "randomized",
) -> pd.Series:
    if search_results_df is not None and len(search_results_df) > 0:
        df = search_results_df.copy()
        if preferred_fold in set(df["fold"].tolist()):
            df = df[df["fold"] == preferred_fold].copy()
        if preferred_search_method in set(df["search_method"].tolist()):
            preferred = df[df["search_method"] == preferred_search_method].copy()
            if len(preferred) > 0:
                df = preferred
        return df.sort_values(["test_auc", "test_f1_score", "test_accuracy"], ascending=False).iloc[0]

    if cv_results_df is not None and len(cv_results_df) > 0:
        df = cv_results_df.copy()
        if preferred_fold in set(df["fold"].tolist()):
            df = df[df["fold"] == preferred_fold].copy()
        return df.sort_values(["cv_auc_mean", "test_auc", "test_f1_score", "test_accuracy"], ascending=False).iloc[0]

    if no_cv_results_df is not None and len(no_cv_results_df) > 0:
        df = no_cv_results_df.copy()
        return df.sort_values(["auc", "f1_score", "accuracy"], ascending=False).iloc[0]

    raise ValueError("Không có kết quả train để chọn model tốt nhất.")


## Cell 7 - Explainability, diagnostics và sinh app Streamlit

**Mục đích:**
- lấy feature importance,
- vẽ heatmap/top pixel,
- chọn model tốt để phân tích,
- tạo ROC/PR/Confusion Matrix,
- tính SHAP,
- sinh file app Streamlit để demo.

**Tham số quan trọng:**
- `TREE_MODEL_NAMES`: chỉ chọn model cây để explainability trực tiếp.
- `TOP_IMPORTANCE_N`: số pixel quan trọng nhất hiển thị.
- `SHAP_SAMPLE_SIZE`: số mẫu dùng cho SHAP.
- `build_streamlit_app_content(...)`: sinh app và chèn rõ câu nhắc rằng đây không phải đánh giá chính thức.


In [12]:
# Cell 7 - Explainability / diagnostics / app helpers
# =========================
# 6) EXPLAINABILITY / FEATURE REDUCTION HELPERS
# =========================
TREE_MODEL_NAMES = ["RandomForest", "LightGBM", "XGBoost"]


def get_model_feature_importance(model: Any) -> np.ndarray:
    if hasattr(model, "feature_importances_"):
        importance = np.asarray(model.feature_importances_, dtype=float)
    elif hasattr(model, "coef_"):
        coef = np.asarray(model.coef_, dtype=float)
        if coef.ndim > 1:
            importance = np.abs(coef).mean(axis=0)
        else:
            importance = np.abs(coef.ravel())
    else:
        raise ValueError(f"Model {type(model).__name__} không hỗ trợ feature importance trực tiếp.")
    return importance.ravel()


def normalize_importance(importance: np.ndarray) -> np.ndarray:
    importance = np.asarray(importance, dtype=float).ravel()
    total = np.sum(np.abs(importance))
    if total <= 0:
        return np.zeros_like(importance)
    return np.abs(importance) / total


def make_importance_df(
    importance: np.ndarray,
    image_shape: Tuple[int, int],
    model_name: str = "model",
    config_name: str = "config",
    stage: str = "cv",
    fold: Optional[int] = None,
) -> pd.DataFrame:
    importance = normalize_importance(importance)
    df = pd.DataFrame({
        "feature_index": np.arange(len(importance), dtype=int),
        "importance": importance,
        "model_name": model_name,
        "config_name": config_name,
        "stage": stage,
        "fold": fold if fold is not None else 0,
    })
    rows, cols = image_shape
    if rows * cols == len(importance):
        df["pixel_row"] = df["feature_index"] // cols
        df["pixel_col"] = df["feature_index"] % cols
    else:
        df["pixel_row"] = 0
        df["pixel_col"] = df["feature_index"]
    return df


def importance_df_to_heatmap(importance_df: pd.DataFrame, image_shape: Tuple[int, int], value_col: str = "importance") -> np.ndarray:
    heat = np.zeros(np.prod(image_shape), dtype=float)
    idx = importance_df["feature_index"].astype(int).to_numpy()
    vals = importance_df[value_col].astype(float).to_numpy()
    heat[idx] = vals
    return heat.reshape(image_shape)


def plot_feature_importance_views(
    importance_df: pd.DataFrame,
    image_shape: Tuple[int, int],
    title: str = "Feature importance",
    top_n: int = 25,
    value_col: str = "importance",
    save_prefix: Optional[str] = None,
) -> None:
    df = importance_df.sort_values(value_col, ascending=False).copy()
    top_df = df.head(min(top_n, len(df))).copy()

    plt.figure(figsize=(9, max(4, top_n * 0.25)))
    y_labels = [f"pix[{int(r)},{int(c)}]" for r, c in zip(top_df["pixel_row"], top_df["pixel_col"])]
    plt.barh(range(len(top_df))[::-1], top_df[value_col].iloc[::-1])
    plt.yticks(range(len(top_df))[::-1], y_labels[::-1])
    plt.xlabel(value_col)
    plt.title(f"{title} - top {len(top_df)} pixels")
    plt.tight_layout()
    if save_prefix is not None:
        plt.savefig(f"{save_prefix}_bar.png", dpi=200, bbox_inches="tight")
    plt.close()

    heat = importance_df_to_heatmap(df, image_shape=image_shape, value_col=value_col)
    plt.figure(figsize=(6, 6))
    plt.imshow(heat, cmap="hot")
    plt.colorbar()
    plt.title(f"{title} - heatmap")
    plt.axis("off")
    plt.tight_layout()
    if save_prefix is not None:
        plt.savefig(f"{save_prefix}_heatmap.png", dpi=200, bbox_inches="tight")
    plt.close()



def select_paper_randomized_source_df(
    randomized_search_results_df: pd.DataFrame,
    preferred_fold: int = 10,
    required_config_name: str = "paper_params",
    required_search_method: str = "randomized",
) -> pd.DataFrame:
    if randomized_search_results_df is None or len(randomized_search_results_df) == 0:
        raise ValueError(
            "Không có randomized_search_results_df. "
            "Hãy bật RUN_RANDOMIZED_SEARCH=True và chạy nhánh RandomizedSearchCV trước."
        )

    df = randomized_search_results_df.copy()

    if "search_method" not in df.columns:
        raise ValueError("Thiếu cột search_method trong randomized_search_results_df.")
    if "config_name" not in df.columns:
        raise ValueError("Thiếu cột config_name trong randomized_search_results_df.")
    if "fold" not in df.columns:
        raise ValueError("Thiếu cột fold trong randomized_search_results_df.")

    df = df[df["search_method"].astype(str).str.lower() == str(required_search_method).lower()].copy()
    df = df[df["config_name"].astype(str) == str(required_config_name)].copy()
    df = df[df["fold"].astype(int) == int(preferred_fold)].copy()

    if len(df) == 0:
        raise ValueError(
            "Không tìm thấy kết quả explainability đúng chuẩn paper "
            f"(search_method={required_search_method}, config_name={required_config_name}, fold={preferred_fold})."
        )

    missing_models = [model_name for model_name in TREE_MODEL_NAMES if model_name not in set(df["model_name"].astype(str).tolist())]
    if missing_models:
        raise ValueError(
            "Thiếu model cây trong nguồn explainability chuẩn paper: "
            + ", ".join(missing_models)
        )

    return df.reset_index(drop=True)


def build_explainability_source_manifest(selected_rows_df: pd.DataFrame) -> pd.DataFrame:
    manifest_df = selected_rows_df.copy()
    manifest_df["explainability_policy"] = "paper_randomized_search_cv10_only"
    manifest_df["explainability_source_note"] = (
        "Only RandomizedSearchCV, fold=10, config_name=paper_params are allowed for feature importance and SHAP."
    )
    return manifest_df.reset_index(drop=True)


def select_tree_rows_for_explainability(
    cv_results_df: Optional[pd.DataFrame] = None,
    search_results_df: Optional[pd.DataFrame] = None,
    no_cv_results_df: Optional[pd.DataFrame] = None,
    preferred_fold: int = 10,
) -> pd.DataFrame:
    selected_rows = []

    source_df = None
    if search_results_df is not None and len(search_results_df) > 0:
        source_df = search_results_df.copy()
        if preferred_fold in set(source_df["fold"].tolist()):
            source_df = source_df[source_df["fold"] == preferred_fold].copy()
    elif cv_results_df is not None and len(cv_results_df) > 0:
        source_df = cv_results_df.copy()
        if preferred_fold in set(source_df["fold"].tolist()):
            source_df = source_df[source_df["fold"] == preferred_fold].copy()
    elif no_cv_results_df is not None and len(no_cv_results_df) > 0:
        source_df = no_cv_results_df.copy()

    if source_df is None or len(source_df) == 0:
        raise ValueError("Không tìm được source dataframe cho explainability.")

    for model_name in TREE_MODEL_NAMES:
        sub = source_df[source_df["model_name"] == model_name].copy()
        if len(sub) == 0:
            continue
        sort_cols = [c for c in ["test_auc", "cv_auc_mean", "auc", "test_f1_score", "f1_score"] if c in sub.columns]
        row = sub.sort_values(sort_cols, ascending=False).iloc[0]
        selected_rows.append(row)

    if len(selected_rows) == 0:
        raise ValueError("Không tìm được tree-based models phù hợp cho explainability.")
    return pd.DataFrame(selected_rows).reset_index(drop=True)


def load_importance_frames(selected_rows_df: pd.DataFrame, image_shape: Tuple[int, int]) -> Tuple[pd.DataFrame, Dict[str, dict]]:
    frames = []
    bundles = {}
    for _, row in selected_rows_df.iterrows():
        bundle = load_model_bundle(row["saved_model_path"])
        model = bundle["model"]
        importance = get_model_feature_importance(model)
        imp_df = make_importance_df(
            importance,
            image_shape=image_shape,
            model_name=row["model_name"],
            config_name=row["config_name"],
            stage=row.get("stage", "cv"),
            fold=row.get("fold", 0),
        )
        key = f"{row['model_name']}__{row['config_name']}__fold_{int(row.get('fold', 0))}"
        bundles[key] = bundle
        frames.append(imp_df)
    return pd.concat(frames, ignore_index=True), bundles


def extract_branch_importance_df(
    importance_frames_df: pd.DataFrame,
    source_row: pd.Series,
) -> pd.DataFrame:
    fold_value = int(source_row.get("fold", 0))
    sub_df = importance_frames_df[
        (importance_frames_df["model_name"] == source_row["model_name"])
        & (importance_frames_df["config_name"] == source_row["config_name"])
        & (importance_frames_df["fold"].astype(int) == fold_value)
    ].copy()
    if len(sub_df) == 0:
        raise ValueError(
            f"Không tìm thấy importance frame cho {source_row['model_name']} | "
            f"{source_row['config_name']} | fold={fold_value}"
        )
    return sub_df.sort_values("importance", ascending=False).reset_index(drop=True)


def select_top_pixels_from_importance_df(
    importance_df: pd.DataFrame,
    top_k: int,
) -> np.ndarray:
    top_k = int(min(top_k, len(importance_df)))
    if top_k <= 0:
        raise ValueError("top_k phải lớn hơn 0.")
    return importance_df.head(top_k)["feature_index"].astype(int).to_numpy()


def reduce_features(X_data: np.ndarray, selected_indices: np.ndarray) -> np.ndarray:
    selected_indices = np.asarray(selected_indices, dtype=int)
    return X_data[:, selected_indices]


def build_reduced_feature_metadata(selected_indices: np.ndarray, total_features: int) -> Dict[str, Any]:
    selected_indices = np.asarray(selected_indices, dtype=int)
    return {
        "top_k_pixels": int(len(selected_indices)),
        "total_features": int(total_features),
        "reduced_feature_ratio": float(len(selected_indices) / max(total_features, 1)),
        "selected_indices_preview": selected_indices[:20].tolist(),
    }


def add_feature_subset_columns(
    df: pd.DataFrame,
    subset_name: str = "all_pixels",
    selected_indices: Optional[np.ndarray] = None,
    total_features: Optional[int] = None,
) -> pd.DataFrame:
    df = df.copy()
    if selected_indices is None:
        df["feature_subset"] = subset_name
        df["top_k_pixels"] = total_features if total_features is not None else np.nan
        df["reduced_feature_ratio"] = 1.0
        return df

    meta = build_reduced_feature_metadata(selected_indices=selected_indices, total_features=total_features)
    df["feature_subset"] = subset_name
    df["top_k_pixels"] = meta["top_k_pixels"]
    df["reduced_feature_ratio"] = meta["reduced_feature_ratio"]
    return df


def select_top_model_rows_for_diagnostics(
    search_results_df: Optional[pd.DataFrame] = None,
    cv_results_df: Optional[pd.DataFrame] = None,
    no_cv_results_df: Optional[pd.DataFrame] = None,
    top_n: int = 3,
    preferred_fold: int = 10,
) -> pd.DataFrame:
    if search_results_df is not None and len(search_results_df) > 0:
        df = search_results_df.copy()
        if preferred_fold in set(df["fold"].tolist()):
            df = df[df["fold"] == preferred_fold].copy()
        df = df.sort_values(["test_auc", "test_f1_score", "test_accuracy"], ascending=False)
        return df.head(top_n).reset_index(drop=True)

    if cv_results_df is not None and len(cv_results_df) > 0:
        df = cv_results_df.copy()
        if preferred_fold in set(df["fold"].tolist()):
            df = df[df["fold"] == preferred_fold].copy()
        df = df.sort_values(["cv_auc_mean", "test_auc", "test_f1_score", "test_accuracy"], ascending=False)
        return df.head(top_n).reset_index(drop=True)

    if no_cv_results_df is not None and len(no_cv_results_df) > 0:
        df = no_cv_results_df.copy()
        df = df.sort_values(["auc", "f1_score", "accuracy"], ascending=False)
        return df.head(top_n).reset_index(drop=True)

    raise ValueError("Không có kết quả train để chọn model diagnostics.")


def plot_model_diagnostics(
    model_row: pd.Series,
    X_eval: np.ndarray,
    y_eval: np.ndarray,
    label_name_map: Optional[dict] = None,
    save_dir: Optional[Path] = None,
    prefix: str = "baseline",
) -> Dict[str, float]:
    label_name_map = label_name_map or LABEL_NAME_MAP
    bundle = load_model_bundle(model_row["saved_model_path"])
    model = bundle["model"]

    model_name = model_row["model_name"]
    config_name = model_row["config_name"]
    fold = int(model_row.get("fold", 0))
    title_stub = f"{prefix} | {model_name} | {config_name} | fold={fold}"

    y_pred = model.predict(X_eval)
    y_score = get_positive_scores(model, X_eval)
    auc_value = roc_auc_score(y_eval, y_score)
    ap_value = average_precision_score(y_eval, y_score)

    display_labels = [decode_label_name(0, label_name_map), decode_label_name(1, label_name_map)]

    plt.figure(figsize=(6, 5))
    RocCurveDisplay.from_predictions(y_eval, y_score)
    plt.title(f"ROC curve - {title_stub}\nAUC={auc_value:.4f}")
    plt.tight_layout()
    if save_dir is not None:
        save_path = Path(save_dir) / f"{sanitize_name(prefix)}_{sanitize_name(model_name)}_{sanitize_name(config_name)}_fold{fold}_roc.png"
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(6, 5))
    PrecisionRecallDisplay.from_predictions(y_eval, y_score)
    plt.title(f"PR curve - {title_stub}\nAP={ap_value:.4f}")
    plt.tight_layout()
    if save_dir is not None:
        save_path = Path(save_dir) / f"{sanitize_name(prefix)}_{sanitize_name(model_name)}_{sanitize_name(config_name)}_fold{fold}_pr.png"
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(5, 5))
    ConfusionMatrixDisplay.from_predictions(y_eval, y_pred, display_labels=display_labels, cmap="Blues")
    plt.title(f"Confusion matrix - {title_stub}")
    plt.tight_layout()
    if save_dir is not None:
        save_path = Path(save_dir) / f"{sanitize_name(prefix)}_{sanitize_name(model_name)}_{sanitize_name(config_name)}_fold{fold}_cm.png"
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()

    return {
        "model_name": model_name,
        "config_name": config_name,
        "fold": fold,
        "auc": float(auc_value),
        "average_precision": float(ap_value),
        "accuracy": float(accuracy_score(y_eval, y_pred)),
        "precision": float(precision_score(y_eval, y_pred, zero_division=0)),
        "recall": float(recall_score(y_eval, y_pred, zero_division=0)),
        "f1_score": float(f1_score(y_eval, y_pred, zero_division=0)),
    }


def try_build_shap_values(model: Any, X_background: np.ndarray, X_sample: np.ndarray) -> Tuple[np.ndarray, Optional[Any]]:
    if not HAS_SHAP:
        raise ImportError("Chưa cài shap. Hãy pip install shap rồi chạy lại cell SHAP.")

    try:
        explainer = shap.Explainer(model, X_background)
        explanation = explainer(X_sample)
        shap_values = np.asarray(explanation.values)
        return shap_values, explanation
    except Exception:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_sample)
        if isinstance(shap_values, list):
            shap_values = shap_values[-1]
        shap_values = np.asarray(shap_values)
        if shap_values.ndim == 3 and shap_values.shape[-1] >= 1:
            shap_values = shap_values[:, :, -1]
        return shap_values, None


def plot_tree_shap_summary(
    model_bundle: dict,
    X_background: np.ndarray,
    X_sample: np.ndarray,
    image_shape: Tuple[int, int],
    title: str = "SHAP summary",
    max_display: int = 20,
    save_prefix: Optional[str] = None,
) -> pd.DataFrame:
    shap_values, explanation = try_build_shap_values(model_bundle["model"], X_background, X_sample)

    plt.figure()
    if explanation is not None:
        shap.plots.bar(explanation, max_display=max_display, show=False)
        plt.title(title)
        plt.tight_layout()
    else:
        shap.summary_plot(shap_values, X_sample, plot_type="bar", max_display=max_display, show=False)
        plt.title(title)
        plt.tight_layout()
    if save_prefix is not None:
        plt.savefig(f"{save_prefix}_shap_bar.png", dpi=200, bbox_inches="tight")
    plt.close()

    if explanation is not None:
        shap_abs_mean = np.abs(np.asarray(explanation.values)).mean(axis=0)
    else:
        shap_abs_mean = np.abs(np.asarray(shap_values)).mean(axis=0)

    shap_df = make_importance_df(
        shap_abs_mean,
        image_shape=image_shape,
        model_name="SHAP",
        config_name="explainability",
        stage="diagnostics",
        fold=0,
    )
    plot_feature_importance_views(
        shap_df,
        image_shape=image_shape,
        title=f"{title} - mean(|SHAP|)",
        top_n=max_display,
        value_col="importance",
        save_prefix=save_prefix,
    )
    return shap_df


def build_streamlit_app_content(default_output_dir: str = "outputs_quishing_revised") -> str:
    return f'''from pathlib import Path
import joblib
import numpy as np
import streamlit as st
import qrcode
from PIL import Image

LABEL_NAME_MAP = {LABEL_NAME_MAP}
SAFE_BENIGN_QR_PAYLOADS = {SAFE_BENIGN_QR_PAYLOADS}
SAFE_SIMULATED_MALICIOUS_QR_PAYLOADS = {SAFE_SIMULATED_MALICIOUS_QR_PAYLOADS}
DEFAULT_OUTPUT_DIR = Path(r"{default_output_dir}")
QR_VERSION = {QR_VERSION}
QR_BORDER = {QR_BORDER}
QR_BOX_SIZE = {QR_BOX_SIZE}
QR_TARGET_SHAPE = {QR_TARGET_SHAPE}

def get_safe_qr_payload(qr_type='benign', sample_index=0):
    qr_type = str(qr_type).strip().lower()
    if qr_type in {{'benign', 'normal', 'safe'}}:
        pool = SAFE_BENIGN_QR_PAYLOADS
        canonical_type = 'benign'
    else:
        pool = SAFE_SIMULATED_MALICIOUS_QR_PAYLOADS
        canonical_type = 'malicious'
    sample_index = int(sample_index) % len(pool)
    return canonical_type, pool[sample_index]

def render_qr_to_array(text, target_shape=QR_TARGET_SHAPE):
    qr = qrcode.QRCode(
        version=QR_VERSION,
        error_correction=qrcode.constants.ERROR_CORRECT_L,
        box_size=QR_BOX_SIZE,
        border=QR_BORDER,
    )
    qr.add_data(text)
    qr.make(fit=False)
    img = qr.make_image(fill_color='black', back_color='white').convert('L')
    arr = np.asarray(img, dtype=np.float32)
    if tuple(arr.shape) != tuple(target_shape):
        raise ValueError(f'QR shape {{arr.shape}} != expected {{target_shape}}')
    return img, arr

def get_positive_scores(model, X):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, 'decision_function'):
        scores = model.decision_function(X)
        scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
        return scores
    return model.predict(X)

def load_model_bundle(path):
    bundle = joblib.load(path)
    if isinstance(bundle, dict) and 'model' in bundle:
        return bundle
    return {{'model': bundle, 'metadata': {{}}}}

def align_generated_array_to_reference(qr_array, reference_stats):
    ref_min = reference_stats.get('min', 0.0)
    ref_max = reference_stats.get('max', 1.0)
    ref_mean = reference_stats.get('mean', 0.5)
    if ref_max <= 1.0 + 1e-12:
        base = qr_array.astype(np.float32) / 255.0 if qr_array.max() > 1 else qr_array.astype(np.float32)
    else:
        base = qr_array.astype(np.float32) / 255.0 * (ref_max - ref_min) + ref_min
    inv = ref_min + ref_max - base
    chosen = base if abs(float(base.mean()) - ref_mean) <= abs(float(inv.mean()) - ref_mean) else inv
    return chosen.astype(np.float32)

st.set_page_config(page_title='QR Quishing Demo', layout='wide')
st.title('QR Quishing Demo')
st.info('Đây là phần minh họa ứng dụng để thử mô hình với QR synthetic an toàn; không phải đánh giá chính thức của mô hình.')

model_files = sorted(DEFAULT_OUTPUT_DIR.glob('models/**/*.joblib'))
if not model_files:
    st.warning('Chưa tìm thấy model .joblib. Hãy chạy script train trước.')
    st.stop()

model_path = st.selectbox('Chọn model đã train', options=[str(p) for p in model_files])
bundle = load_model_bundle(model_path)
model = bundle['model']
metadata = bundle.get('metadata', {{}})
reference_stats = metadata.get('reference_image_stats', {{'min': 0.0, 'max': 1.0, 'mean': 0.5}})

col1, col2 = st.columns(2)
with col1:
    input_mode = st.radio('Nguồn QR', ['Synthetic', 'Text nhập tay'])
    qr_type = st.selectbox('Loại QR synthetic', ['benign', 'malicious'])
    sample_index = st.number_input('Sample index', min_value=0, max_value=20, value=0, step=1)
    custom_text = st.text_area('Nội dung QR thủ công', value='https://example.com')
with col2:
    try:
        if input_mode == 'Synthetic':
            qr_type, text = get_safe_qr_payload(qr_type=qr_type, sample_index=sample_index)
        else:
            text = custom_text
            qr_type = 'custom'

        img, arr = render_qr_to_array(text=text, target_shape=QR_TARGET_SHAPE)
        arr = align_generated_array_to_reference(arr, reference_stats)
        X_input = arr.reshape(1, -1)
        pred_label = int(model.predict(X_input)[0])
        score_1 = float(get_positive_scores(model, X_input)[0])

        st.image(img, caption=f'QR rendered | source={{qr_type}} | version=13 | 69x69 | EC=LOW | border=0', use_container_width=False)
        st.write('**Payload**')
        st.code(text)
        st.write('**Prediction**')
        st.json({{
            'predicted_label': pred_label,
            'predicted_label_name': LABEL_NAME_MAP.get(pred_label, f'class_{{pred_label}}'),
            'prob_class_1': round(score_1, 6),
            'selected_model': model_path,
        }})
    except Exception as e:
        st.error(str(e))

st.caption('Lưu ý: đây là minh họa ứng dụng với QR synthetic an toàn; phần đánh giá chính thức vẫn phải dựa trên test set và cross-validation.')
'''


def write_streamlit_demo_app(output_path: Path, default_output_dir: str = "outputs_quishing_revised") -> Path:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(build_streamlit_app_content(default_output_dir=default_output_dir), encoding="utf-8")
    return output_path


## Cell 8 - Nạp dữ liệu và chia train/test

**Mục đích:**
- tìm file pickle ảnh QR và label,
- chuyển về `numpy`,
- flatten ảnh QR thành vector đặc trưng,
- chia train/test theo 80/20,
- xuất bảng glossary giải thích metric.

**Đầu ra chính:** `X_train`, `X_test`, `y_train`, `y_test`, `image_shape`.


In [13]:
# Cell 8 - Load data và train/test split
# =========================
# 7) MAIN PIPELINE
# =========================
qr_path = resolve_existing_path(QR_IMAGE_CANDIDATES)
label_path = resolve_existing_path(QR_LABEL_CANDIDATES)

qr_images = load_pickle(qr_path)
labels = load_pickle(label_path)

qr_images = ensure_numpy(qr_images)
labels = ensure_numpy(labels).astype(int)

X, image_shape = flatten_qr_images(qr_images)
y = labels

print("QR file   :", qr_path)
print("Label file:", label_path)
print("QR shape  :", qr_images.shape)
print("X shape   :", X.shape)
print("y shape   :", y.shape)
print("Image size:", image_shape)
print("Class dist:", pd.Series(y).value_counts().sort_index().to_dict())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

metric_glossary_df = build_metric_glossary_df()
metric_glossary_df.to_csv(RESULT_DIR / "metric_glossary.csv", index=False)

# 7.1) EDA
eda_summary = pd.DataFrame({
    "n_samples": [len(y)],
    "n_features": [X.shape[1]],
    "image_height": [image_shape[0]],
    "image_width": [image_shape[1]],
    "positive_ratio": [float(np.mean(y))],
    "negative_ratio": [float(1 - np.mean(y))],
    "pixel_min": [float(np.min(X))],
    "pixel_max": [float(np.max(X))],
    "pixel_mean": [float(np.mean(X))],
    "pixel_std": [float(np.std(X))],
})
eda_summary.to_csv(EDA_DIR / "eda_summary.csv", index=False)


QR file   : dataset\qr_codes_29.pickle
Label file: dataset\qr_codes_29_labels.pickle
QR shape  : (9987, 69, 69)
X shape   : (9987, 4761)
y shape   : (9987,)
Image size: (69, 69)
Class dist: {0: 5005, 1: 4982}
Train shape: (7989, 4761)
Test shape : (1998, 4761)


## Cell 9 - EDA

**Mục đích:** khảo sát dữ liệu trước khi train.

**Những gì cell này làm:**
- tạo `eda_summary.csv`,
- vẽ phân bố lớp,
- hiển thị một số QR mẫu,
- thống kê mật độ pixel,
- so sánh QR trung bình theo từng lớp.

**Ý nghĩa:** giúp bạn viết phần mô tả dataset và trực quan hóa cho báo cáo.


In [14]:
# Cell 9 - EDA

class_counts = pd.Series(y).map({0: "Benign", 1: "Phishing"}).value_counts()
ax = class_counts.plot(kind="bar", title="Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(EDA_DIR / "class_distribution.png", dpi=150, bbox_inches="tight")
plt.close()

if qr_images.ndim == 3:
    n_samples = min(10, len(qr_images))
    indices = np.random.choice(len(qr_images), size=n_samples, replace=False)
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    axes = axes.flatten()
    for ax, idx in zip(axes, indices):
        ax.imshow(qr_images[idx], cmap="gray")
        ax.set_title(f"Label={labels[idx]}")
        ax.axis("off")
    for ax in axes[n_samples:]:
        ax.axis("off")
    plt.suptitle("Sample QR Codes")
    plt.tight_layout()
    plt.savefig(EDA_DIR / "sample_qr_codes.png", dpi=150, bbox_inches="tight")
    plt.close()

pixel_density = pd.DataFrame({
    "label": y,
    "mean_pixel_intensity": X.mean(axis=1),
    "active_pixel_ratio": (X > 0).mean(axis=1),
})
pixel_density.groupby("label").agg(["mean", "std", "min", "max"]).to_csv(EDA_DIR / "pixel_density_summary.csv")

plt.figure(figsize=(8, 5))
for label_value, label_name in [(0, "Benign"), (1, "Phishing")]:
    plt.hist(
        pixel_density.loc[pixel_density["label"] == label_value, "mean_pixel_intensity"],
        bins=30,
        alpha=0.6,
        label=label_name,
    )
plt.title("Distribution of Mean Pixel Intensity")
plt.xlabel("Mean Pixel Intensity")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.savefig(EDA_DIR / "mean_pixel_intensity_hist.png", dpi=150, bbox_inches="tight")
plt.close()

if qr_images.ndim == 3:
    benign_mean = qr_images[y == 0].mean(axis=0)
    phishing_mean = qr_images[y == 1].mean(axis=0)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(benign_mean, cmap="gray")
    axes[0].set_title("Mean QR - Benign")
    axes[0].axis("off")
    axes[1].imshow(phishing_mean, cmap="gray")
    axes[1].set_title("Mean QR - Phishing")
    axes[1].axis("off")
    plt.tight_layout()
    plt.savefig(EDA_DIR / "mean_qr_by_class.png", dpi=150, bbox_inches="tight")
    plt.close()

# 7.2) Models


## Cell 10 - Khởi tạo model set và biến chứa kết quả

**Mục đích:** tạo `model_sets`, `search_spaces` và các DataFrame rỗng để gom kết quả qua từng stage.


In [15]:
# Cell 10 - Initialize models và result holders
model_sets, search_spaces = build_model_sets(random_state=RANDOM_STATE)

# Stage holders
no_cv_results_df = pd.DataFrame()
cv_results_df = pd.DataFrame()
grid_search_results_df = pd.DataFrame()
randomized_search_results_df = pd.DataFrame()
reduced_no_cv_results_df = pd.DataFrame()
reduced_cv_results_df = pd.DataFrame()


## Cell 11 - Huấn luyện không cross-validation

**Mục đích:** train nhanh trên `X_train`, đánh giá trên `X_test`, lưu model và xuất `no_cv_results.csv`.

**Khi nào nên bật:**
- cần baseline nhanh,
- cần so sánh với các stage nặng hơn như CV/Search.


In [16]:
# Cell 11 - Train without CV
if RUN_NO_CV:
    no_cv_results_df = train_without_cv(
        model_sets=model_sets,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        model_dir=MODEL_DIR,
    ).sort_values(["auc", "f1_score", "accuracy"], ascending=False).reset_index(drop=True)
    no_cv_results_df.to_csv(RESULT_DIR / "no_cv_results.csv", index=False)



===== TRAIN NO-CV | CONFIG: paper_params =====
LogisticRegression | AUC=0.8798 | F1=0.7942 | saved=LogisticRegression.joblib
DecisionTree       | AUC=0.8211 | F1=0.7656 | saved=DecisionTree.joblib
RandomForest       | AUC=0.8966 | F1=0.7970 | saved=RandomForest.joblib
GaussianNB         | AUC=0.6791 | F1=0.4801 | saved=GaussianNB.joblib
LightGBM           | AUC=0.9164 | F1=0.8320 | saved=LightGBM.joblib
XGBoost            | AUC=0.9132 | F1=0.8305 | saved=XGBoost.joblib

===== TRAIN NO-CV | CONFIG: custom_params =====
LogisticRegression | AUC=0.8704 | F1=0.7840 | saved=LogisticRegression.joblib
DecisionTree       | AUC=0.7588 | F1=0.7588 | saved=DecisionTree.joblib
RandomForest       | AUC=0.8982 | F1=0.8007 | saved=RandomForest.joblib
GaussianNB         | AUC=0.6791 | F1=0.4801 | saved=GaussianNB.joblib
LightGBM           | AUC=0.9164 | F1=0.8320 | saved=LightGBM.joblib
XGBoost            | AUC=0.9115 | F1=0.8278 | saved=XGBoost.joblib


## Cell 12 - Huấn luyện có cross-validation

**Mục đích:** so sánh độ ổn định của mô hình với các fold `5, 10, 15, 20`.

**Đầu ra:** `cv_results_all_folds.csv` và các model đã fit lại trên toàn bộ train set.


In [17]:
# Cell 12 - Train with CV
if RUN_CV:
    cv_results_df = train_with_cv(
        model_sets=model_sets,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        cv_folds=CV_FOLDS,
        model_dir=MODEL_DIR,
    ).sort_values(["fold", "cv_auc_mean", "test_auc"], ascending=[True, False, False]).reset_index(drop=True)
    cv_results_df.to_csv(RESULT_DIR / "cv_results_all_folds.csv", index=False)



================ CV = 5 ================

----- CONFIG: paper_params -----
LogisticRegression | CV AUC=0.8778 ± 0.0095 | Test AUC=0.8798
DecisionTree       | CV AUC=0.8227 ± 0.0070 | Test AUC=0.8211
RandomForest       | CV AUC=0.8983 ± 0.0093 | Test AUC=0.8966
GaussianNB         | CV AUC=0.6735 ± 0.0091 | Test AUC=0.6791
LightGBM           | CV AUC=0.9158 ± 0.0086 | Test AUC=0.9164
XGBoost            | CV AUC=0.9139 ± 0.0064 | Test AUC=0.9132

----- CONFIG: custom_params -----
LogisticRegression | CV AUC=0.8672 ± 0.0104 | Test AUC=0.8704
DecisionTree       | CV AUC=0.7431 ± 0.0137 | Test AUC=0.7588
RandomForest       | CV AUC=0.9001 ± 0.0074 | Test AUC=0.8982
GaussianNB         | CV AUC=0.6735 ± 0.0091 | Test AUC=0.6791
LightGBM           | CV AUC=0.9158 ± 0.0086 | Test AUC=0.9164
XGBoost            | CV AUC=0.9132 ± 0.0071 | Test AUC=0.9115

================ CV = 10 ================

----- CONFIG: paper_params -----
LogisticRegression | CV AUC=0.8798 ± 0.0120 | Test AUC=0.8798
Decisi

## Cell 13 - GridSearchCV và RandomizedSearchCV thật sự

**Mục đích:** tối ưu siêu tham số thật cho từng mô hình theo `SEARCH_SCORING`.

**Lưu ý tham số:**
- `SEARCH_SCORING = 'f1'`: tối ưu theo F1-score.
- `SEARCH_CV_FOLD = 10`: search dùng 10-fold.
- `RANDOM_SEARCH_N_ITER = 12`: số tổ hợp ngẫu nhiên tối đa cho RandomizedSearchCV.

**Đầu ra:**
- `grid_search_results.csv`
- `randomized_search_results.csv`
- `search_results_combined.csv`


In [18]:
# Cell 13 - Run GridSearchCV / RandomizedSearchCV

if RUN_GRID_SEARCH:
    grid_search_results_df = train_with_search(
        model_sets=model_sets,
        search_spaces=search_spaces,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        search_cv_fold=SEARCH_CV_FOLD,
        method="grid",
        model_dir=MODEL_DIR,
        scoring=SEARCH_SCORING,
    ).sort_values(["test_auc", "test_f1_score", "test_accuracy"], ascending=False).reset_index(drop=True)
    grid_search_results_df.to_csv(RESULT_DIR / "grid_search_results.csv", index=False)

if RUN_RANDOMIZED_SEARCH:
    randomized_search_results_df = train_with_search(
        model_sets=model_sets,
        search_spaces=search_spaces,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        search_cv_fold=SEARCH_CV_FOLD,
        method="randomized",
        model_dir=MODEL_DIR,
        scoring=SEARCH_SCORING,
    ).sort_values(["test_auc", "test_f1_score", "test_accuracy"], ascending=False).reset_index(drop=True)
    randomized_search_results_df.to_csv(RESULT_DIR / "randomized_search_results.csv", index=False)


================ GRID SEARCH | CV = 10 ================

----- CONFIG: paper_params -----
LogisticRegression | best_f1=0.7916 | Test AUC=0.8858 | params={'C': 0.01, 'penalty': 'l2', 'solver': 'lbfgs'}
DecisionTree       | best_f1=0.7692 | Test AUC=0.8211 | params={'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}
RandomForest       | best_f1=0.7890 | Test AUC=0.9006 | params={'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 400}
GaussianNB         | best_f1=0.4918 | Test AUC=0.6844 | params={'var_smoothing': 1e-08}
LightGBM           | best_f1=0.8348 | Test AUC=0.9135 | params={'learning_rate': 0.1, 'max_depth': -1, 'n_estimators': 200, 'num_leaves': 63}
XGBoost            | best_f1=0.8338 | Test AUC=0.9159 | params={'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 400, 'subsample': 1.0}

----- CONFIG: custom_params -----
LogisticRegression | best_f1=0.7920 | Test AUC=0.8859 | params={'C': 0.01, 'penalty': 'l2',

## Cell 14 - Bảng tổng hợp cho báo cáo và môi trường thực nghiệm

**Mục đích:**
- tạo các pivot table để đưa vào báo cáo,
- tạo bảng **Môi trường thực nghiệm** gồm Python, sklearn, xgboost, lightgbm, CPU, RAM, thời gian train.

**File quan trọng:** `experiment_environment.csv`.


In [ ]:
# Cell 14 - Report tables + experiment environment



# 7.3) Tables for report
if len(no_cv_results_df) > 0:
    no_cv_auc_pivot = no_cv_results_df.pivot_table(index="model_name", columns="config_name", values="auc").sort_index()
    no_cv_auc_pivot.to_csv(RESULT_DIR / "no_cv_auc_pivot.csv")

if len(cv_results_df) > 0:
    cv_auc_pivot = cv_results_df.pivot_table(index=["fold", "model_name"], columns="config_name", values="cv_auc_mean").sort_index()
    cv_auc_pivot.to_csv(RESULT_DIR / "cv_auc_pivot.csv")
    best_by_fold = cv_results_df.sort_values(["fold", "cv_auc_mean", "test_auc"], ascending=[True, False, False]).groupby("fold", as_index=False).first()
    best_by_fold.to_csv(RESULT_DIR / "best_model_by_fold.csv", index=False)

if len(grid_search_results_df) > 0 or len(randomized_search_results_df) > 0:
    search_compare_df = pd.concat([grid_search_results_df, randomized_search_results_df], ignore_index=True, sort=False)
    search_compare_df.to_csv(RESULT_DIR / "search_results_combined.csv", index=False)
    search_pivot = search_compare_df.pivot_table(index=["model_name", "config_name"], columns="search_method", values="test_auc")
    search_pivot.to_csv(RESULT_DIR / "search_auc_pivot.csv")
else:
    search_compare_df = pd.DataFrame()

# 7.4) Environment table
stage_frames = {
    "no_cv": no_cv_results_df,
    "cv": cv_results_df,
    "grid_search": grid_search_results_df,
    "randomized_search": randomized_search_results_df,
}
experiment_env_df = collect_experiment_environment(
    image_shape=image_shape,
    X_train=X_train,
    X_test=X_test,
    stage_frames=stage_frames,
)

## Cell 15 - Explainability và thí nghiệm rút gọn đặc trưng

**Mục đích:**
- chọn 3 model cây tốt nhất để lấy feature importance,
- **khóa chặt nguồn explainability theo đúng paper**:
  - chỉ lấy từ **RandomizedSearchCV**
  - chỉ lấy **fold = 10**
  - chỉ lấy nhánh **`paper_params`**
- tạo **3 nhánh Feature Selection riêng biệt** theo đúng paper:
  - FS based on **Random Forest**
  - FS based on **LightGBM**
  - FS based on **XGBoost**
- với mỗi nhánh, giữ lại `TOP_K_PIXELS` pixel quan trọng nhất của **chính model nguồn**,
- train lại toàn bộ tập mô hình trên từng bộ đặc trưng rút gọn,
- so sánh với baseline **No FS**.

**Ý nghĩa báo cáo:** phần này bám sát pipeline paper hơn và tránh lẫn với `grid_search` hoặc `custom_params`.


In [20]:
# Cell 15 - Explainability + reduced feature experiments
experiment_env_df.to_csv(RESULT_DIR / "experiment_environment.csv", index=False)

# 7.5) Explainability + reduced features
if RUN_REDUCED_FEATURE_EXPERIMENTS:
    explain_source_df = select_paper_randomized_source_df(
        randomized_search_results_df=randomized_search_results_df,
        preferred_fold=EXPLAINABILITY_SOURCE_FOLD,
        required_config_name=EXPLAINABILITY_SOURCE_CONFIG_NAME,
        required_search_method=EXPLAINABILITY_SOURCE_SEARCH_METHOD,
    )

    tree_rows_df = select_tree_rows_for_explainability(
        cv_results_df=None,
        search_results_df=explain_source_df,
        no_cv_results_df=None,
        preferred_fold=EXPLAINABILITY_SOURCE_FOLD,
    )
    tree_rows_df = build_explainability_source_manifest(tree_rows_df)
    tree_rows_df.to_csv(RESULT_DIR / "tree_models_for_explainability.csv", index=False)
    explain_source_df.to_csv(RESULT_DIR / "paper_randomized_search_cv10_source.csv", index=False)

    importance_frames_df, _ = load_importance_frames(selected_rows_df=tree_rows_df, image_shape=image_shape)
    importance_frames_df.to_csv(RESULT_DIR / "tree_feature_importances_long.csv", index=False)

    feature_branch_meta_rows = []
    reduced_no_cv_frames = []
    reduced_cv_frames = []

    top_k_pixels = int(min(TOP_K_PIXELS, X_train.shape[1]))

    for _, row in tree_rows_df.iterrows():
        fold_value = int(row.get("fold", 0))
        source_model_name = str(row["model_name"])
        source_config_name = str(row["config_name"])
        feature_subset_name = f"fs_{sanitize_name(source_model_name).lower()}"

        branch_importance_df = extract_branch_importance_df(
            importance_frames_df=importance_frames_df,
            source_row=row,
        )
        branch_importance_path = RESULT_DIR / f"{feature_subset_name}_feature_importance.csv"
        branch_importance_df.to_csv(branch_importance_path, index=False)

        save_prefix = EXPLAIN_DIR / f"{sanitize_name(source_model_name)}_{sanitize_name(source_config_name)}_fold{fold_value}"
        plot_feature_importance_views(
            importance_df=branch_importance_df,
            image_shape=image_shape,
            title=f"Feature importance | {source_model_name} | {source_config_name} | fold={fold_value}",
            top_n=TOP_IMPORTANCE_N,
            value_col="importance",
            save_prefix=str(save_prefix),
        )

        selected_pixel_indices = select_top_pixels_from_importance_df(
            importance_df=branch_importance_df,
            top_k=top_k_pixels,
        )
        np.save(RESULT_DIR / f"{feature_subset_name}_selected_pixel_indices.npy", selected_pixel_indices)

        X_train_reduced = reduce_features(X_train, selected_pixel_indices)
        X_test_reduced = reduce_features(X_test, selected_pixel_indices)

        branch_meta = build_reduced_feature_metadata(
            selected_indices=selected_pixel_indices,
            total_features=X_train.shape[1],
        )
        branch_meta.update({
            "feature_subset": feature_subset_name,
            "fs_source_model": source_model_name,
            "fs_source_config": source_config_name,
            "fs_source_fold": fold_value,
            "source_stage": row.get("stage", "cv"),
            "source_search_method": row.get("search_method", "none"),
            "explainability_policy": row.get("explainability_policy", "paper_randomized_search_cv10_only"),
            "branch_importance_path": str(branch_importance_path),
        })
        feature_branch_meta_rows.append(branch_meta)

        branch_model_dir = MODEL_DIR / "feature_selection" / feature_subset_name

        branch_no_cv_df = train_without_cv(
            model_sets=model_sets,
            X_train=X_train_reduced,
            X_test=X_test_reduced,
            y_train=y_train,
            y_test=y_test,
            model_dir=branch_model_dir,
        )
        branch_no_cv_df = add_feature_subset_columns(
            branch_no_cv_df,
            subset_name=feature_subset_name,
            selected_indices=selected_pixel_indices,
            total_features=X_train.shape[1],
        )
        branch_no_cv_df["fs_source_model"] = source_model_name
        branch_no_cv_df["fs_source_config"] = source_config_name
        branch_no_cv_df["fs_source_fold"] = fold_value
        branch_no_cv_df["branch_importance_path"] = str(branch_importance_path)
        branch_no_cv_df["explainability_policy"] = row.get("explainability_policy", "paper_randomized_search_cv10_only")
        branch_no_cv_df = branch_no_cv_df.sort_values(["auc", "f1_score", "accuracy"], ascending=False).reset_index(drop=True)
        branch_no_cv_df.to_csv(RESULT_DIR / f"{feature_subset_name}_no_cv_results.csv", index=False)
        reduced_no_cv_frames.append(branch_no_cv_df)

        branch_cv_df = train_with_cv(
            model_sets=model_sets,
            X_train=X_train_reduced,
            X_test=X_test_reduced,
            y_train=y_train,
            y_test=y_test,
            cv_folds=CV_FOLDS,
            model_dir=branch_model_dir,
        )
        branch_cv_df = add_feature_subset_columns(
            branch_cv_df,
            subset_name=feature_subset_name,
            selected_indices=selected_pixel_indices,
            total_features=X_train.shape[1],
        )
        branch_cv_df["fs_source_model"] = source_model_name
        branch_cv_df["fs_source_config"] = source_config_name
        branch_cv_df["fs_source_fold"] = fold_value
        branch_cv_df["branch_importance_path"] = str(branch_importance_path)
        branch_cv_df["explainability_policy"] = row.get("explainability_policy", "paper_randomized_search_cv10_only")
        branch_cv_df = branch_cv_df.sort_values(["fold", "cv_auc_mean", "test_auc"], ascending=[True, False, False]).reset_index(drop=True)
        branch_cv_df.to_csv(RESULT_DIR / f"{feature_subset_name}_cv_results_all_folds.csv", index=False)
        reduced_cv_frames.append(branch_cv_df)

    feature_branch_meta_df = pd.DataFrame(feature_branch_meta_rows)
    feature_branch_meta_df.to_csv(RESULT_DIR / "feature_selection_branch_metadata.csv", index=False)

    baseline_no_cv_cmp = add_feature_subset_columns(
        no_cv_results_df,
        subset_name="no_fs",
        selected_indices=None,
        total_features=X_train.shape[1],
    )
    baseline_no_cv_cmp["fs_source_model"] = "No FS"
    baseline_no_cv_cmp["fs_source_config"] = "all_pixels"
    baseline_no_cv_cmp["fs_source_fold"] = 0
    baseline_no_cv_cmp["branch_importance_path"] = ""
    baseline_no_cv_cmp["explainability_policy"] = "baseline_all_pixels"

    baseline_cv_cmp = add_feature_subset_columns(
        cv_results_df,
        subset_name="no_fs",
        selected_indices=None,
        total_features=X_train.shape[1],
    )
    baseline_cv_cmp["fs_source_model"] = "No FS"
    baseline_cv_cmp["fs_source_config"] = "all_pixels"
    baseline_cv_cmp["fs_source_fold"] = 0
    baseline_cv_cmp["branch_importance_path"] = ""
    baseline_cv_cmp["explainability_policy"] = "baseline_all_pixels"

    reduced_no_cv_results_df = pd.concat(reduced_no_cv_frames, ignore_index=True, sort=False) if len(reduced_no_cv_frames) > 0 else pd.DataFrame()
    reduced_cv_results_df = pd.concat(reduced_cv_frames, ignore_index=True, sort=False) if len(reduced_cv_frames) > 0 else pd.DataFrame()

    compare_no_cv_frames = [baseline_no_cv_cmp]
    if len(reduced_no_cv_results_df) > 0:
        compare_no_cv_frames.append(reduced_no_cv_results_df)

    compare_cv_frames = [baseline_cv_cmp]
    if len(reduced_cv_results_df) > 0:
        compare_cv_frames.append(reduced_cv_results_df)

    compare_no_cv_df = pd.concat(compare_no_cv_frames, ignore_index=True, sort=False)
    compare_cv_df = pd.concat(compare_cv_frames, ignore_index=True, sort=False)

    compare_no_cv_df.to_csv(RESULT_DIR / "compare_no_fs_vs_feature_selection_no_cv.csv", index=False)
    compare_cv_df.to_csv(RESULT_DIR / "compare_no_fs_vs_feature_selection_cv.csv", index=False)

    compare_no_cv_summary_df = compare_no_cv_df.sort_values(
        ["feature_subset", "auc", "f1_score", "accuracy"],
        ascending=[True, False, False, False],
    ).reset_index(drop=True)
    compare_cv_summary_df = compare_cv_df.sort_values(
        ["feature_subset", "fold", "cv_auc_mean", "test_auc"],
        ascending=[True, True, False, False],
    ).reset_index(drop=True)

    compare_no_cv_summary_df.to_csv(RESULT_DIR / "compare_no_fs_vs_feature_selection_no_cv_sorted.csv", index=False)
    compare_cv_summary_df.to_csv(RESULT_DIR / "compare_no_fs_vs_feature_selection_cv_sorted.csv", index=False)



===== TRAIN NO-CV | CONFIG: paper_params =====
LogisticRegression | AUC=0.8753 | F1=0.7871 | saved=LogisticRegression.joblib
DecisionTree       | AUC=0.8183 | F1=0.7618 | saved=DecisionTree.joblib
RandomForest       | AUC=0.9034 | F1=0.8202 | saved=RandomForest.joblib
GaussianNB         | AUC=0.8228 | F1=0.6994 | saved=GaussianNB.joblib
LightGBM           | AUC=0.9070 | F1=0.8320 | saved=LightGBM.joblib
XGBoost            | AUC=0.9068 | F1=0.8186 | saved=XGBoost.joblib

===== TRAIN NO-CV | CONFIG: custom_params =====
LogisticRegression | AUC=0.8727 | F1=0.7859 | saved=LogisticRegression.joblib
DecisionTree       | AUC=0.7457 | F1=0.7413 | saved=DecisionTree.joblib
RandomForest       | AUC=0.9045 | F1=0.8241 | saved=RandomForest.joblib
GaussianNB         | AUC=0.8228 | F1=0.6994 | saved=GaussianNB.joblib
LightGBM           | AUC=0.9070 | F1=0.8320 | saved=LightGBM.joblib
XGBoost            | AUC=0.9067 | F1=0.8304 | saved=XGBoost.joblib

================ CV = 5 ================

----- 

## Cell 16 - Diagnostics, SHAP, sinh app demo và QR demo

**Mục đích:**
- chọn top model để vẽ ROC/PR/Confusion Matrix,
- tính SHAP cho model cây tốt nhất,
- sinh file app Streamlit,
- chạy demo QR synthetic với model tốt nhất,
- gom toàn bộ kết quả vào `full_training_report.csv`.

**Nhấn mạnh:** app/demo này chỉ là **minh họa ứng dụng**, không phải benchmark chính thức.


In [21]:
# Cell 16 - Diagnostics + SHAP + Streamlit app + QR demo + full report

# 7.6) Diagnostics
combined_search_df = pd.concat(
    [df for df in [randomized_search_results_df, grid_search_results_df] if df is not None and len(df) > 0],
    ignore_index=True,
    sort=False,
) if (len(randomized_search_results_df) > 0 or len(grid_search_results_df) > 0) else pd.DataFrame()

diagnostic_rows_df = select_top_model_rows_for_diagnostics(
    search_results_df=combined_search_df,
    cv_results_df=cv_results_df,
    no_cv_results_df=no_cv_results_df,
    top_n=TOP_DIAGNOSTIC_MODELS,
    preferred_fold=PREFERRED_DEMO_FOLD,
)

diagnostic_summary_rows = []
for _, row in diagnostic_rows_df.iterrows():
    diagnostic_summary_rows.append(
        plot_model_diagnostics(
            model_row=row,
            X_eval=X_test,
            y_eval=y_test,
            label_name_map=LABEL_NAME_MAP,
            save_dir=PLOT_DIR,
            prefix="best_models",
        )
    )

diagnostic_summary_df = pd.DataFrame(diagnostic_summary_rows)
diagnostic_summary_df.to_csv(RESULT_DIR / "diagnostic_summary.csv", index=False)

# 7.7) SHAP
if HAS_SHAP:
    shap_source_df = select_paper_randomized_source_df(
        randomized_search_results_df=randomized_search_results_df,
        preferred_fold=EXPLAINABILITY_SOURCE_FOLD,
        required_config_name=EXPLAINABILITY_SOURCE_CONFIG_NAME,
        required_search_method=EXPLAINABILITY_SOURCE_SEARCH_METHOD,
    )
    if len(shap_source_df) > 0:
        sort_cols = [c for c in ["test_auc", "cv_auc_mean", "auc", "test_f1_score", "f1_score"] if c in shap_source_df.columns]
        best_tree_candidates = shap_source_df[shap_source_df["model_name"].isin(TREE_MODEL_NAMES)].copy()
        if len(best_tree_candidates) > 0:
            best_tree_row = best_tree_candidates.sort_values(sort_cols, ascending=False).iloc[0]
            best_tree_bundle = load_model_bundle(best_tree_row["saved_model_path"])
            background_size = int(min(SHAP_SAMPLE_SIZE, len(X_train)))
            sample_size = int(min(SHAP_SAMPLE_SIZE, len(X_test)))
            X_background = X_train[:background_size]
            X_sample = X_test[:sample_size]
            shap_importance_df = plot_tree_shap_summary(
                model_bundle=best_tree_bundle,
                X_background=X_background,
                X_sample=X_sample,
                image_shape=image_shape,
                title=f"SHAP | {best_tree_row['model_name']} | {best_tree_row['config_name']} | fold={int(best_tree_row.get('fold', 0))}",
                max_display=TOP_IMPORTANCE_N,
                save_prefix=str(EXPLAIN_DIR / f"shap_{sanitize_name(best_tree_row['model_name'])}_{sanitize_name(best_tree_row['config_name'])}"),
            )
            shap_importance_df.to_csv(RESULT_DIR / "shap_importance_mean_abs.csv", index=False)

# 7.8) Build Streamlit app
streamlit_app_path = write_streamlit_demo_app(
    output_path=APP_DIR / "app_qr_demo_streamlit.py",
    default_output_dir=str(OUTPUT_DIR),
)

# 7.9) QR synthetic demo using best available model
best_model_row = select_best_model_row(
    no_cv_results_df=no_cv_results_df,
    cv_results_df=cv_results_df,
    search_results_df=combined_search_df,
    preferred_fold=PREFERRED_DEMO_FOLD,
    preferred_search_method=PREFERRED_DEMO_SEARCH_METHOD,
)

best_model_bundle = load_model_bundle(best_model_row["saved_model_path"])
best_model_bundle.setdefault("metadata", {})["reference_image_stats"] = {
    "min": float(np.min(qr_images)),
    "max": float(np.max(qr_images)),
    "mean": float(np.mean(qr_images)),
}
save_trained_model(
    best_model_bundle["model"],
    Path(best_model_row["saved_model_path"]),
    metadata=best_model_bundle["metadata"],
)

generated_qr_demo_rows = []
for qr_type in ["benign", "malicious"]:
    for sample_index in [0, 1]:
        result = predict_generated_qr(
            model_bundle=best_model_bundle,
            reference_images=qr_images,
            target_shape=image_shape,
            qr_type=qr_type,
            sample_index=sample_index,
            label_name_map=LABEL_NAME_MAP,
            save_dir=GENERATED_QR_DIR,
            show_plot=False,
        )
        generated_qr_demo_rows.append(result)

generated_qr_demo_df = pd.DataFrame(generated_qr_demo_rows)
generated_qr_demo_df.to_csv(RESULT_DIR / "generated_qr_demo_predictions.csv", index=False)

# 7.10) Full report merge
report_frames = []
if len(no_cv_results_df) > 0:
    tmp = no_cv_results_df.copy()
    for col in ["accuracy", "precision", "recall", "f1_score", "auc"]:
        tmp[f"test_{col}"] = tmp[col]
    for col in ["cv_accuracy_mean", "cv_precision_mean", "cv_recall_mean", "cv_f1_score_mean", "cv_auc_mean", "search_scoring", "search_best_score", "search_best_params"]:
        tmp[col] = np.nan
    report_frames.append(tmp)
if len(cv_results_df) > 0:
    tmp = cv_results_df.copy()
    for col in ["search_scoring", "search_best_score", "search_best_params"]:
        tmp[col] = np.nan
    report_frames.append(tmp)
if len(grid_search_results_df) > 0:
    report_frames.append(grid_search_results_df.copy())
if len(randomized_search_results_df) > 0:
    report_frames.append(randomized_search_results_df.copy())

if len(report_frames) > 0:
    full_report_df = pd.concat(report_frames, ignore_index=True, sort=False)
    full_report_df.to_csv(RESULT_DIR / "full_training_report.csv", index=False)

print("\n========== DONE ==========")
print("Saved results dir:", RESULT_DIR.resolve())
print("Saved models dir :", MODEL_DIR.resolve())
print("Saved app        :", streamlit_app_path.resolve())
print("\nFiles nổi bật:")
print("- experiment_environment.csv")
print("- no_cv_results.csv")
print("- cv_results_all_folds.csv")
print("- grid_search_results.csv")
print("- randomized_search_results.csv")
print("- full_training_report.csv")
print("- generated_qr_demo_predictions.csv")
print("\nLưu ý: app/demo QR chỉ là minh họa ứng dụng, không phải đánh giá chính thức.")



========== DONE ==========
Saved results dir: C:\Users\httnghia\Downloads\ATTT_ML\outputs\results
Saved models dir : C:\Users\httnghia\Downloads\ATTT_ML\outputs\models
Saved app        : C:\Users\httnghia\Downloads\ATTT_ML\outputs\app\app_qr_demo_streamlit.py

Files nổi bật:
- experiment_environment.csv
- no_cv_results.csv
- cv_results_all_folds.csv
- grid_search_results.csv
- randomized_search_results.csv
- full_training_report.csv
- generated_qr_demo_predictions.csv

Lưu ý: app/demo QR chỉ là minh họa ứng dụng, không phải đánh giá chính thức.


<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

## Kết quả đầu ra quan trọng sau khi chạy xong

Bạn nên kiểm tra các file sau trong thư mục `outputs_quishing_revised/results/`:
- `experiment_environment.csv`
- `no_cv_results.csv`
- `cv_results_all_folds.csv`
- `grid_search_results.csv`
- `randomized_search_results.csv`
- `search_results_combined.csv`
- `full_training_report.csv`
- `generated_qr_demo_predictions.csv`

Ngoài ra, app Streamlit sẽ nằm ở `outputs_quishing_revised/app/app_qr_demo_streamlit.py`.
